# Cardiovascular Risk Intelligence & Segmentation Platform EDA

This notebook explores the raw and harmonized Framingham + cardio datasets.

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from src.ingest import DataIngestor
from src.features import FeatureEngineer

base_dir = Path.cwd().parents[0] if (Path.cwd() / 'src').exists() is False else Path.cwd()
raw_dir = base_dir / 'data' / 'raw'
framingham_raw = pd.read_csv(raw_dir / 'heart_disease.csv')
cardio_raw = pd.read_csv(raw_dir / 'cardio_train.csv', sep=';')
ingestor = DataIngestor(base_dir)
harmonized = ingestor.harmonize()
featured = FeatureEngineer().engineer(harmonized)


## Section 1: Dataset Overview

In [ ]:
display(framingham_raw.shape, cardio_raw.shape)
display(framingham_raw.dtypes, cardio_raw.dtypes)
display(framingham_raw.isna().sum(), cardio_raw.isna().sum())
display(framingham_raw['TenYearCHD'].value_counts(normalize=True), cardio_raw['cardio'].value_counts(normalize=True))


## Section 2: Outlier Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.boxplot(y=cardio_raw['ap_hi'], ax=axes[0])
sns.boxplot(y=cardio_raw['ap_lo'], ax=axes[1])
cardio_raw['bmi'] = cardio_raw['weight'] / ((cardio_raw['height'] / 100) ** 2)
sns.boxplot(y=cardio_raw['bmi'], ax=axes[2])
plt.tight_layout()
print('Before cleaning:', len(framingham_raw), len(cardio_raw))
print('After cleaning:', len(harmonized[harmonized['source']=='framingham']), len(harmonized[harmonized['source']=='cardio']))


## Section 3: Feature Distributions

In [ ]:
for col in ['age_years', 'bmi', 'systolic_bp']:
    plt.figure(figsize=(6, 4))
    sns.kdeplot(data=featured, x=col, hue='target', common_norm=False)
    plt.show()
sns.histplot(data=featured, x='lifestyle_risk_score', hue='target', multiple='dodge')
plt.show()


## Section 4: Risk Stratification Analysis

In [ ]:
display(featured.groupby('age_group')['target'].mean())
display(featured.groupby('bp_category')['target'].mean())
display(featured.groupby('gender_bin')['target'].mean())
featured['lifestyle_bucket'] = pd.cut(featured['lifestyle_risk_score'], bins=[-1, 1, 2, 4], labels=['0-1', '2', '3-4'])
display(featured.groupby('lifestyle_bucket')['target'].mean())


## Section 5: Correlation Analysis

In [ ]:
corr = featured[['age_years','bmi','systolic_bp','diastolic_bp','pulse_pressure','cholesterol_raw','glucose_raw','lifestyle_risk_score','bp_bmi_interaction','age_bp_interaction','target']].corr(numeric_only=True)
plt.figure(figsize=(10, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.show()
print(corr['target'].drop('target').abs().sort_values(ascending=False).head(5))


## Section 6: Cross-Dataset Comparison

In [ ]:
for feature in ['systolic_bp', 'age_years', 'bmi']:
    plt.figure(figsize=(6, 4))
    sns.kdeplot(data=featured, x=feature, hue='source', common_norm=False)
    plt.show()
print('Comment on distribution differences between the Framingham and cardio cohorts here.')


## Section 7: Insight Summary

In [ ]:
normal_risk = featured.loc[featured['bp_category']==0, 'target'].mean()
stage2_risk = featured.loc[featured['bp_category']==3, 'target'].mean()
high_lifestyle = featured.loc[featured['lifestyle_risk_score']>2, 'target'].mean()
low_lifestyle = featured.loc[featured['lifestyle_risk_score']<=2, 'target'].mean()
senior_positive = featured[(featured['age_group']==2) & (featured['target']==1)]
pulse_pressure_share = (senior_positive['pulse_pressure']>60).mean()*100 if len(senior_positive) else 0
senior_male = featured[(featured['age_group']==2) & (featured['gender_bin']==1)]['target'].mean()
senior_female = featured[(featured['age_group']==2) & (featured['gender_bin']==0)]['target'].mean()
print(f"Stage 2 hypertension patients have {stage2_risk / normal_risk:.2f}x the cardio event rate of Normal BP patients")
print(f"Lifestyle risk score >2 correlates with {((high_lifestyle - low_lifestyle) / low_lifestyle * 100):.1f}% higher incidence")
print(f"Pulse pressure >60 mmHg appears in {pulse_pressure_share:.1f}% of cardio-positive seniors")
print(f"Gender modifies risk differently by age group: senior males show {senior_male:.2f}x vs senior females {senior_female:.2f}x")
